# Quadrature velocity moving-window simulation

This notebook generates a 0.8 Hz encoder signal with the production generator, creates the four-edge quadrature signal with the production processor, and compares representative pulse-window settings over 10 seconds. It also implements the time-difference-first method used in `recoder/h5viewer.ipynb`.

In [ ]:
from pathlib import Path
import queue
import threading

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lib_ipmu_daq_config import AppConfig, _loadConfig
from lib_ipmu_daq_generator import Generator
from lib_ipmu_daq_process import PULSES_PER_REVOLUTION, Processor

INPUT_VELOCITY_HZ = 0.8
DURATION_SEC = 10.0

preset = _loadConfig(Path("_config_preset.toml"))
run_config = _loadConfig(Path("_config_run.toml"))
preset["debug_encoder"]["input_velocity"] = INPUT_VELOCITY_HZ
cfg = AppConfig.fromDict(preset, run_config)
SAMPLE_RATE_HZ = cfg.io.sample_rate

settings = [
    (256, 192),
    (128, 64),
    (64, 32),
    (32, 8),
    (32, 16),
    (16, 4),
    (16, 1),
]

print(f"Sample rate: {SAMPLE_RATE_HZ:,} Hz")
print(f"Duration: {DURATION_SEC:.1f} s")
print(f"Expected quadrature event rate: {INPUT_VELOCITY_HZ * PULSES_PER_REVOLUTION:.1f} pulses/s")

## 1. Generate the 0.8 Hz A/B phase signals

`Generator._genChunkPulse` is used directly so that pulse width, duty cycle, height, and phase follow the production generator.

In [ ]:
sample_count = int(SAMPLE_RATE_HZ * DURATION_SEC)
time_axis = np.arange(sample_count, dtype=np.float64) / SAMPLE_RATE_HZ

generator = Generator(cfg, buf_q=None, stop_event=None)
pulse_A = generator._genChunkPulse(time_axis, phase=cfg.debug_encoder.pulse_phase_A)
pulse_B = generator._genChunkPulse(time_axis, phase=cfg.dependent.pulse_phase_B)

print(f"Generated {len(time_axis):,} samples for each phase.")

In [ ]:
preview_end_sec = 0.012
preview = time_axis <= preview_end_sec

fig, ax = plt.subplots(figsize=(12, 4))
ax.step(time_axis[preview] * 1e3, pulse_A[preview], where="post", label="Phase A")
ax.step(time_axis[preview] * 1e3, pulse_B[preview], where="post", label="Phase B")
ax.set(title="Generated A/B phase signals at 0.8 Hz", xlabel="Time [ms]", ylabel="Amplitude [V]")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 2. Create the four-edge quadrature signal

The production `_getPulseDirection` method detects every A/B edge. `_genQuadPulse` converts the signed edge log into the pulse waveform used by the application.

In [ ]:
def make_processor(config):
    return Processor(
        config=config,
        buf_q=queue.Queue(),
        quad_q=queue.Queue(),
        comvel_q=queue.Queue(),
        DataStoreFlag=queue.Queue(),
        stop_event=threading.Event(),
        debug=False,
    )

quadrature_processor = make_processor(cfg)
direction_log, last_A, last_B = quadrature_processor._getPulseDirection(
    pulse_A,
    pulse_B,
    threshold=cfg.encoder_postproc.threshold,
)
quadrature_signal = quadrature_processor._genQuadPulse(time_axis, direction_log)
event_count = np.count_nonzero(direction_log)

print(f"Detected quadrature events: {event_count:,}")
print(f"Measured event rate: {event_count / DURATION_SEC:.1f} pulses/s")
print(f"Whole-record velocity: {direction_log.sum() / DURATION_SEC / PULSES_PER_REVOLUTION:.6f} Hz")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].step(time_axis[preview] * 1e3, direction_log[preview], where="post", color="black")
axes[0].set(ylabel="Direction event", title="Four-edge quadrature events")
axes[1].plot(time_axis[preview] * 1e3, quadrature_signal[preview], color="tab:purple")
axes[1].set(xlabel="Time [ms]", ylabel="Amplitude [V]", title="Generated quadrature pulse waveform")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Run the production pulse-window estimator

Each setting is evaluated with a fresh `Processor`, preserving the production startup zero-fill and overlap semantics: `step = window - overlap`.

In [ ]:
production_results = {}
production_summary = []

for window_pulses, overlap_pulses in settings:
    cfg.encoder_postproc.movingave_window_pulses = window_pulses
    cfg.encoder_postproc.movingave_overlap_pulses = overlap_pulses
    processor = make_processor(cfg)
    result_time, result_velocity = processor._getVelocityMovingAve(time_axis, direction_log)
    label = f"{window_pulses}/{overlap_pulses}"
    production_results[label] = (result_time, result_velocity)

    stable_velocity = result_velocity[result_velocity != 0.0]
    stable_time = result_time[result_velocity != 0.0]
    point_interval_ms = np.median(np.diff(stable_time)) * 1e3 if len(stable_time) > 1 else np.nan
    production_summary.append({
        "setting": label,
        "window": window_pulses,
        "overlap": overlap_pulses,
        "step": window_pulses - overlap_pulses,
        "output_points": len(result_velocity),
        "median_interval_ms": point_interval_ms,
        "median_velocity_hz": np.median(stable_velocity) if len(stable_velocity) else np.nan,
    })

production_summary_df = pd.DataFrame(production_summary)
production_summary_df

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True, sharey=True)
for ax, (label, (result_time, result_velocity)) in zip(axes.flat, production_results.items()):
    ax.plot(result_time, result_velocity, color="tab:red", linewidth=1.0)
    ax.axhline(INPUT_VELOCITY_HZ, color="black", linestyle="--", linewidth=1.0)
    ax.set_title(f"window/overlap = {label}")
    ax.set(xlabel="Time [s]", ylabel="Velocity [Hz]", xlim=(0, DURATION_SEC))
    ax.grid(True, alpha=0.3)
for ax in axes.flat[len(production_results):]:
    ax.set_visible(False)
fig.suptitle("Production pulse-window velocity estimates")
plt.tight_layout()
plt.show()

## 4. Calculate per-event velocity from time differences

Following the approach in `h5viewer.ipynb`, quadrature event times are extracted first and velocity is calculated from consecutive event-time differences.

In [ ]:
event_indices = np.flatnonzero(direction_log)
event_times = time_axis[event_indices]
event_directions = direction_log[event_indices].astype(np.float64)
event_dt = np.diff(event_times)
dt_times = event_times[1:]
dt_velocity = event_directions[1:] / event_dt / PULSES_PER_REVOLUTION

print(f"Time-difference velocity points: {len(dt_velocity):,}")
print(f"Median velocity: {np.median(dt_velocity):.6f} Hz")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
show_dt = dt_times <= 0.2
ax.plot(dt_times[show_dt], dt_velocity[show_dt], marker=".", linestyle="none", markersize=3)
ax.axhline(INPUT_VELOCITY_HZ, color="black", linestyle="--")
ax.set(title="Per-event velocity from quadrature time differences", xlabel="Time [s]", ylabel="Velocity [Hz]")
ax.grid(True, alpha=0.3)
plt.show()

## 5. Apply the same moving-window settings after time differencing

This method takes the arithmetic mean of the per-event velocity values. It is intentionally separate from the production estimator, which divides the signed pulse count by the total window elapsed time.

In [ ]:
def moving_average_after_time_difference(times, values, window_pulses, overlap_pulses):
    step = window_pulses - overlap_pulses
    kernel = np.ones(window_pulses, dtype=np.float64) / window_pulses
    rolling_velocity = np.convolve(values, kernel, mode="valid")
    output_indices = np.arange(0, len(rolling_velocity), step)
    output_times = times[output_indices + window_pulses - 1]
    return output_times, rolling_velocity[output_indices]

time_difference_results = {}
time_difference_summary = []

for window_pulses, overlap_pulses in settings:
    result_time, result_velocity = moving_average_after_time_difference(
        dt_times, dt_velocity, window_pulses, overlap_pulses
    )
    label = f"{window_pulses}/{overlap_pulses}"
    time_difference_results[label] = (result_time, result_velocity)
    time_difference_summary.append({
        "setting": label,
        "window": window_pulses,
        "overlap": overlap_pulses,
        "step": window_pulses - overlap_pulses,
        "output_points": len(result_velocity),
        "median_interval_ms": np.median(np.diff(result_time)) * 1e3,
        "median_velocity_hz": np.median(result_velocity),
    })

time_difference_summary_df = pd.DataFrame(time_difference_summary)
time_difference_summary_df

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True, sharey=True)
for ax, (label, (result_time, result_velocity)) in zip(axes.flat, time_difference_results.items()):
    ax.plot(result_time, result_velocity, color="tab:blue", linewidth=1.0)
    ax.axhline(INPUT_VELOCITY_HZ, color="black", linestyle="--", linewidth=1.0)
    ax.set_title(f"window/overlap = {label}")
    ax.set(xlabel="Time [s]", ylabel="Velocity [Hz]", xlim=(0, DURATION_SEC))
    ax.grid(True, alpha=0.3)
for ax in axes.flat[len(time_difference_results):]:
    ax.set_visible(False)
fig.suptitle("Moving average applied after per-event time differencing")
plt.tight_layout()
plt.show()

## 6. Compare both estimators

The production estimator and the time-difference-first estimator are overlaid for every window/overlap setting.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True, sharey=True)
for ax, label in zip(axes.flat, production_results):
    prod_time, prod_velocity = production_results[label]
    diff_time, diff_velocity = time_difference_results[label]
    stable = prod_velocity != 0.0
    ax.plot(prod_time[stable], prod_velocity[stable], color="tab:red", linewidth=1.0, label="Production pulse span")
    ax.plot(diff_time, diff_velocity, color="tab:blue", linewidth=1.0, alpha=0.75, label="Mean after time difference")
    ax.axhline(INPUT_VELOCITY_HZ, color="black", linestyle="--", linewidth=1.0)
    ax.set_title(f"window/overlap = {label}")
    ax.set(xlabel="Time [s]", ylabel="Velocity [Hz]", xlim=(0, 1.0))
    ax.grid(True, alpha=0.3)
axes.flat[0].legend()
for ax in axes.flat[len(production_results):]:
    ax.set_visible(False)
fig.suptitle("Estimator comparison: first second after startup")
plt.tight_layout()
plt.show()